## 7. Mech Interp 1

In [ ]:
# from huggingface_hub import hf_hub_download, notebook_login
# notebook_login()

In [ ]:
# !pip install "torch>=2.1.0" einops datasets jaxtyping "sae_lens==3.23.1" openai tabulate "nbformat>=4.2.0" umap-learn hdbscan eindex-callum git+https://github.com/callummcdougall/CircuitsVis.git#subdirectory=python git+https://github.com/callummcdougall/sae_vis.git@callum/v3 "transformer-lens>=2.7.0"

In [ ]:
from IPython.display import HTML, IFrame, clear_output, display
import random
from rich import print as rprint
from rich.table import Table
from tqdm.auto import tqdm
from functools import partial
import torch as t
import torch.nn.functional as F
from torch import Tensor, nn
from jaxtyping import Float, Int
import einops

import matplotlib.pyplot as plt
import numpy as np

device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")

from sae_lens.toolkit.pretrained_saes_directory import get_pretrained_saes_directory
from sae_lens import (
    SAE,
    ActivationsStore,
    HookedSAETransformer,
    LanguageModelSAERunnerConfig,
    SAEConfig,
    SAETrainingRunner,
    upload_saes_to_huggingface,
)

from transformer_lens import ActivationCache, HookedTransformer, utils
from transformer_lens.hook_points import HookPoint
from tabulate import tabulate

import pickle, glob

In [ ]:
gemmascope_sae_release = "gemma-scope-2b-pt-mlp-canonical"
gemmascope_sae_id = "layer_20/width_16k/canonical"

In [ ]:
gemma_2_2b = HookedSAETransformer.from_pretrained("gemma-2-2b", device=device)

In [ ]:
gemma_2_2b_sae = SAE.from_pretrained(gemmascope_sae_release, gemmascope_sae_id, device=str(device))[0]

In [ ]:
prompt = "The reliability of Wikipedia is very"
answer = " high"

In [ ]:
utils.test_prompt(prompt, answer, gemma_2_2b)

In [ ]:
tokens=gemma_2_2b.to_tokens(prompt)
logits, cache = gemma_2_2b.run_with_cache(tokens, remove_batch_dim=True)

In [ ]:
#Now doing this mapping manually, should match outputs above, just less cool printout, do I need softcap here?
probs=logits[:,-1,:].softmax(dim=-1)
token_probs, token_ids = probs[-1, :].topk(10, dim=-1)

for tk, p in zip(token_ids, token_probs):
  print(gemma_2_2b.to_string(tk.item()), round(p.item(), 4))

## Longer Generation

In [ ]:
prompt_2= "The reliability of Wikipedia is very important"

In [ ]:
GENERATE_KWARGS = dict(temperature=0.5, freq_penalty=2.0, verbose=False)
gemma_2_2b.generate(prompt_2, max_new_tokens=50, **GENERATE_KWARGS)

In [ ]:
# Intrudction tuned Gemma
# gemma_2_2b_it = HookedSAETransformer.from_pretrained("gemma-2-2b-it", device=device)

## Visualize Data Moving Through model

In [ ]:
prompt = "The reliability of Wikipedia is very"

In [ ]:
tokens=gemma_2_2b.to_tokens(prompt)
logits, cache = gemma_2_2b.run_with_cache(tokens, remove_batch_dim=True)

In [ ]:
cache['hook_embed'].shape

In [ ]:
t.testing.assert_close(cache['hook_embed'], cache['blocks.0.hook_resid_pre'])

In [ ]:
m=cache['hook_embed'].detach().cpu().numpy()[-1,:].reshape(48,48)

In [ ]:
fig=plt.figure(0, (12,12))
ax=fig.add_subplot(111)
plt.imshow(m); 
#plt.colorbar()

for i in range(m.shape[0]):
    for j in range(m.shape[1]):
        ax.text(
            j, i, f"{m[i, j]:.1f}",
            ha="center", va="center",
            color="white", fontsize=5.5, 
            fontname="cmr10",
        )
plt.axis('off')
plt.savefig('data/book_exports/7_mech_interp/gemma_layer_1.png', dpi=600, bbox_inches='tight')

In [ ]:
fig=plt.figure(0, (12,12))
ax=fig.add_subplot(111)
plt.imshow(m); 
#plt.colorbar()

for i in range(m.shape[0]):
    for j in range(m.shape[1]):
        ax.text(
            j, i, f"{m[i, j]:.1f}",
            ha="center", va="center",
            color="white", fontsize=5.5, 
            fontname="cmr10",
        )
plt.axis('off')
plt.savefig('data/book_exports/7_mech_interp/gemma_layer_1.png', dpi=600, bbox_inches='tight')

In [ ]:
m=cache['ln_final.hook_normalized'].detach().cpu().numpy()[-1,:].reshape(48,48)

fig=plt.figure(0, (12,12))
ax=fig.add_subplot(111)
plt.imshow(m); 
#plt.colorbar()

for i in range(m.shape[0]):
    for j in range(m.shape[1]):
        ax.text(
            j, i, f"{m[i, j]:.1f}",
            ha="center", va="center",
            color="white", fontsize=5.5,fontname="cmr10"
        )
plt.axis('off')
plt.savefig('data/book_exports/7_mech_interp/gemma_final_layer_1.png', dpi=600, bbox_inches='tight')

In [ ]:
m=cache['blocks.25.hook_resid_post'].detach().cpu().numpy()[-1,:].reshape(48,48)

fig=plt.figure(0, (12,12))
ax=fig.add_subplot(111)
plt.imshow(m); 
#plt.colorbar()

for i in range(m.shape[0]):
    for j in range(m.shape[1]):
        ax.text(
            j, i, f"{m[i, j]:.1f}",
            ha="center", va="center",
            color="white", fontsize=5.5,fontname="cmr10"
        )
plt.axis('off')
plt.savefig('data/book_exports/7_mech_interp/gemma_final_layer_b.png', dpi=600, bbox_inches='tight')

In [ ]:
for layer_id in range(26):
    m=cache['blocks.'+str(layer_id)+'.hook_resid_pre'].detach().cpu().numpy()[-1,:].reshape(48,48)
    
    plt.clf()
    fig=plt.figure(0, (4,4))
    ax=fig.add_subplot(111)
    plt.imshow(m); 
    plt.axis('off')
    plt.savefig('data/book_exports/7_mech_interp/hook_resid_pre'+str(layer_id).zfill(2)+'.png', dpi=600, bbox_inches='tight')

In [ ]:
for layer_id in range(26):
    m=cache['blocks.'+str(layer_id)+'.hook_resid_post'].detach().cpu().numpy()[-1,:].reshape(48,48)
    
    plt.clf()
    fig=plt.figure(0, (4,4))
    ax=fig.add_subplot(111)
    plt.imshow(m); 
    plt.axis('off')
    plt.savefig('data/book_exports/7_mech_interp/hook_resid_post_'+str(layer_id).zfill(2)+'.png', dpi=600, bbox_inches='tight')

## Work on table 

In [ ]:
#Before model
activations_norm_ln_final=gemma_2_2b.ln_final(cache['hook_embed'])
m=activations_norm_ln_final.detach().cpu().numpy()[-1,:]
plt.imshow(m.reshape(48,48)); plt.colorbar()

In [ ]:
my_logits = gemma_2_2b.unembed(activations_norm_ln_final)
my_logits_softcap = gemma_2_2b.cfg.output_logits_soft_cap * F.tanh(my_logits / gemma_2_2b.cfg.output_logits_soft_cap)
# my_logits_softcap=my_logits

probs=my_logits_softcap.softmax(dim=-1)
token_probs, token_ids = probs[-1, :].topk(10, dim=-1)
for tk, p in zip(token_ids, token_probs):
  print(gemma_2_2b.to_string(tk.item()), round(p.item(), 4))

In [ ]:
tokens=gemma_2_2b.to_tokens(prompt)
logits, cache = gemma_2_2b.run_with_cache(tokens, remove_batch_dim=True)
all_res=[]

for layer_id in range(25):
  activations_norm_next_layer=cache['blocks.'+str(layer_id+1)+'.ln1.hook_normalized']
  # activations_norm_ln_final=gemma_2_2b.ln_final(activations)

  my_logits = gemma_2_2b.unembed(activations_norm_next_layer)
  my_logits_softcap = gemma_2_2b.cfg.output_logits_soft_cap * F.tanh(my_logits / gemma_2_2b.cfg.output_logits_soft_cap)
  # my_logits_softcap=my_logits

  probs=my_logits_softcap.softmax(dim=-1)
  token_probs, token_ids = probs[-1, :].topk(10, dim=-1)
  res={}
  for tk, p in zip(token_ids, token_probs):
    res[gemma_2_2b.to_string(tk.item())] = round(p.item(), 4)
  all_res.append(res)
  # print(layer_id, res)

#Final outputs
probs=logits.softmax(dim=-1)
token_probs, token_ids = probs[0,-1, :].topk(10, dim=-1)
res={}
for tk, p in zip(token_ids, token_probs):
  res[gemma_2_2b.to_string(tk.item())] = round(p.item(), 4)
all_res.append(res)
# print(layer_id+1, res)

In [ ]:
all_res

In [ ]:
import pandas as pd
import json

keys_data = []
for d in all_res:
    # Add quotes around each token to preserve spacing visibility
    row = [f'"{key}"' for key in d.keys()]
    keys_data.append(row)

# Extract values (probabilities)
values_data = []
for d in all_res:
    row = list(d.values())
    values_data.append(row)

# Create DataFrames
# For keys, we need to handle variable column counts
max_cols = max(len(row) for row in keys_data)
keys_df = pd.DataFrame(keys_data, columns=[f'Token_{i+1}' for i in range(max_cols)])

# For values
values_df = pd.DataFrame(values_data, columns=[f'Prob_{i+1}' for i in range(max_cols)])

# Save to CSV
keys_df.to_csv('data/book_exports/7_mech_interp/tokens_keys_2.csv', index=False)
values_df.to_csv('data/book_exports/7_mech_interp/tokens_values_2.csv', index=False)

### Output we "do surgery on"

In [ ]:
layer_id=20
subplot_rows=1

residual_stream=cache['blocks.'+str(layer_id)+'.hook_resid_pre'].detach().cpu().numpy()
attn_out=cache['blocks.'+str(layer_id)+'.hook_attn_out'].detach().cpu().numpy()
residual_stream_mid=cache['blocks.'+str(layer_id)+'.hook_resid_mid'].detach().cpu().numpy()
mlp_out=cache['blocks.'+str(layer_id)+'.hook_mlp_out'].detach().cpu().numpy()

fig=plt.figure(0, (16,8))
plt.subplot(subplot_rows,4,1); plt.imshow(residual_stream[-1].reshape(48,48)); plt.colorbar(aspect=8); plt.title('Pre Layer '+str(layer_id)+' residual stream', fontsize=8); plt.xticks([]); plt.yticks([]);
plt.subplot(subplot_rows,4,2); plt.imshow(attn_out[-1].reshape(48,48)); plt.colorbar(aspect=8); plt.title('Layer '+str(layer_id)+' attention out', fontsize=8); plt.xticks([]); plt.yticks([]);
plt.subplot(subplot_rows,4,3); plt.imshow(residual_stream_mid[-1].reshape(48,48)); plt.colorbar(aspect=8); plt.title('Layer '+str(layer_id)+' residual mid', fontsize=8); plt.xticks([]); plt.yticks([]);
plt.subplot(subplot_rows,4,4); plt.imshow(mlp_out[-1].reshape(48,48)); plt.colorbar(aspect=8); plt.title('Layer '+str(layer_id)+' mlp out', fontsize=8); plt.xticks([]); plt.yticks([]);

In [ ]:
fig=plt.figure(0, (12,12))
ax=fig.add_subplot(111)
m=mlp_out[-1].reshape(48,48) #use mlp output to insepct activations

plt.imshow(m); 
#plt.colorbar()

for i in range(m.shape[0]):
    for j in range(m.shape[1]):
        ax.text(
            j, i, f"{m[i, j]:.1f}",
            ha="center", va="center",
            color="white", fontsize=5.5, 
            fontname="cmr10",
        )
plt.axis('off')
plt.savefig('data/book_exports/7_mech_interp/layer_20_out.png', dpi=600, bbox_inches='tight')

In [ ]:
mlp_out

In [ ]:
next_layer_normed=cache['blocks.'+str(21)+'.ln1.hook_normalized'][-1].detach().cpu().numpy().reshape(48,48) #After layernorm
plt.imshow(next_layer_normed)

In [ ]:
# cache

## Steering

In [ ]:
neuron_index=1393

In [ ]:
def scale_hook(value, hook):
    value[:, :, neuron_index] = -160
    return value

logits = gemma_2_2b.run_with_hooks(tokens, fwd_hooks=[('blocks.20.hook_mlp_out', scale_hook)])
probs=logits[:,-1,:].softmax(dim=-1)
token_probs, token_ids = probs[-1, :].topk(10, dim=-1)
for tk, p in zip(token_ids, token_probs):
    print(gemma_2_2b.to_string(tk.item()), round(p.item(), 4))

In [ ]:
def scale_hook(value, hook):
    value[:, :, neuron_index] = 160
    return value

logits = gemma_2_2b.run_with_hooks(tokens, fwd_hooks=[('blocks.20.hook_mlp_out', scale_hook)])
probs=logits[:,-1,:].softmax(dim=-1)
token_probs, token_ids = probs[-1, :].topk(10, dim=-1)
for tk, p in zip(token_ids, token_probs):
    print(gemma_2_2b.to_string(tk.item()), round(p.item(), 4))

## SAE

In [ ]:
def display_dashboard(
    sae_release="gpt2-small-res-jb",
    sae_id="blocks.7.hook_resid_pre",
    latent_idx=0,
    width=1200,
    height=800,
):
    release = get_pretrained_saes_directory()[sae_release]
    neuronpedia_id = release.neuronpedia_id[sae_id]

    url = f"https://neuronpedia.org/{neuronpedia_id}/{latent_idx}?embed=true&embedexplanation=true&embedplots=true&embedtest=true&height=300"

    print(url)
    display(IFrame(url, width=width, height=height))

def steering_hook(
    activations: Float[Tensor, "batch pos d_in"],
    hook: HookPoint,
    sae: SAE,
    latent_idx: int,
    steering_coefficient: float,
) -> Tensor:
    """
    Steers the model by returning a modified activations tensor, with some multiple of the steering vector added to all
    sequence positions.
    """
    return activations + steering_coefficient * sae.W_dec[latent_idx] #Hmm wow how can we visualize this - interesting. Pretty simple in a way.


def generate_with_steering( #Ah I could do a version of this for neurons - right!
    model: HookedSAETransformer,
    sae: SAE,
    prompt: str,
    latent_idx: int,
    steering_coefficient: float = 1.0,
    max_new_tokens: int = 50,
):
    """
    Generates text with steering. A multiple of the steering vector (the decoder weight for this latent) is added to
    the last sequence position before every forward pass.
    """
    _steering_hook = partial(
        steering_hook,
        sae=sae,
        latent_idx=latent_idx,
        steering_coefficient=steering_coefficient,
    )

    with model.hooks(fwd_hooks=[(sae.cfg.hook_name, _steering_hook)]):
        output = model.generate(prompt, max_new_tokens=max_new_tokens, **GENERATE_KWARGS)

    return output

In [ ]:
_, cache = gemma_2_2b.run_with_cache_with_saes(prompt, saes=[gemma_2_2b_sae])

for name, param in cache.items():
    if "hook_sae" in name:
        print(f"{name:<43}: {tuple(param.shape)}")

In [ ]:
# Get top activations on final token
_, cache = gemma_2_2b.run_with_cache_with_saes(
    prompt,
    saes=[gemma_2_2b_sae],
    stop_at_layer=gemma_2_2b_sae.cfg.hook_layer + 1,
)
sae_acts_post = cache[f"{gemma_2_2b_sae.cfg.hook_name}.hook_sae_acts_post"][0, -1, :]

In [ ]:
m=cache['blocks.20.hook_mlp_out.hook_sae_input']
# m=cache['blocks.20.attn.hook_z.hook_sae_input']
m.shape

In [ ]:
plt.imshow(m[0, -1, :].detach().cpu().numpy().reshape(48,48))

In [ ]:
m=cache['blocks.20.hook_mlp_out.hook_sae_acts_post'].detach().cpu().numpy()
# m=cache['blocks.20.attn.hook_z.hook_sae_acts_post']
m.shape

In [ ]:
plt.imshow(m[0, -1, :].reshape(128,128))

In [ ]:
top_indices = np.argsort(np.abs(m[0,-1]))[-10:][::-1]
m[0,-1][top_indices]

In [ ]:
top_indices

In [ ]:
display_dashboard(sae_release=gemmascope_sae_release, sae_id=gemmascope_sae_id, latent_idx=8249)

In [ ]:
# for act, ind in zip(*sae_acts_post.topk(5)):
#     print(f"Latent {ind} had activation {act:.2f}")
#     # display_dashboard(latent_idx=ind)
#     display_dashboard(sae_release=gemmascope_sae_release, sae_id=gemmascope_sae_id, latent_idx=ind)

In [ ]:
GENERATE_KWARGS = dict(temperature=0.5, freq_penalty=2.0, verbose=False)

In [ ]:
# Test our prompt, to see what the model says
with gemma_2_2b.saes(saes=[gemma_2_2b_sae]):
    utils.test_prompt(prompt, answer, gemma_2_2b)

In [ ]:
latent_idx=8249

In [ ]:
_steering_hook = partial(
    steering_hook,
    sae=gemma_2_2b_sae,
    latent_idx=latent_idx,
    steering_coefficient=100,
)

with gemma_2_2b.hooks(fwd_hooks=[(gemma_2_2b_sae.cfg.hook_name, _steering_hook)]):
  utils.test_prompt(prompt, answer, gemma_2_2b)

In [ ]:
_steering_hook = partial(
    steering_hook,
    sae=gemma_2_2b_sae,
    latent_idx=latent_idx,
    steering_coefficient=500,
)

with gemma_2_2b.hooks(fwd_hooks=[(gemma_2_2b_sae.cfg.hook_name, _steering_hook)]):
  utils.test_prompt(prompt, answer, gemma_2_2b)

In [ ]:
mlp_out=cache['blocks.'+str(20)+'.hook_mlp_out.hook_sae_output'][0, -1].detach().cpu().numpy()
plt.imshow(mlp_out.reshape(48,48)); plt.axis('off')
plt.savefig('data/book_exports/7_mech_interp/post_sae_layer_20_mlp.png', dpi=600, bbox_inches='tight')